# W06B 실습 A · 고유값부터 SVD 전체 계산까지

딥러닝응용I(추천시스템) · 동덕여자대학교 데이터사이언스전공 · 유원상 교수 · 2026-2

2026년 10월 8일. 목표: 행렬곱→고유방향→AᵀA→특이값→U→재구성을 직접 계산하고 검산합니다. 수업 중 1–4절을 함께, 확장 활동은 수업 후 진행합니다.


## 실행과 코드 읽기

CPU 런타임에서 첫 셀부터 순서대로 실행합니다. GPU·업로드·외부 데이터는 필요 없습니다. 파일 → Drive에 사본 저장으로 자신의 실습본을 만드세요. 설치 후 재시작 안내가 뜨면 재시작 후 첫 셀부터 다시 실행합니다.

빈칸 변수와 자동 실험은 분리되어 있습니다. 전체 실행 성공은 자신의 빈칸·서술 답이 정답이라는 뜻이 아닙니다. `None`과 빈 문자열을 자신의 계산·설명으로 바꾸세요. 수치 배열의 미세한 오차는 정상이며 `np.allclose`로 비교합니다.

외부에서 가져온 수업 함수는 아래 링크로 정의 행과 설명을 바로 확인합니다. 함수명 → docstring의 입력/출력 → 핵심 수식 → return 순서로 읽으세요. NumPy 공식 문서: [svd](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html), [eigh](https://numpy.org/doc/stable/reference/generated/numpy.linalg.eigh.html).

| 함수 | 입력 | 출력·사용 이유 |
|---|---|---|
| teaching_matrix | 예제 이름 | 수정 가능한 작은 행렬 사본 |
| unit_vector | 0이 아닌 벡터 | 같은 방향, 길이1 |
| symmetric_eigen | 실수 대칭행렬 | 내림차순 values, 열별 vectors, 대입 잔차 |
| svd_steps | 행렬, 남길 rank | U/s/Vt/Gram/근사/오차/에너지 사전 |
| transform_stages | 2×2 행렬, 2×N 점 | 입력→Vt→Sigma→U의 네 배열 |
| direction_view | 예제, 각도 | 설명·표·화살표 그림 |
| stages_view | 예제 | 설명·단위원 변환 그림 |
| reconstruction_view | 예제, rank, 선택 report_fn | 설명·근사 표·비교 그림 |
| build_svd_app | 선택 report_fn | 웹 앱 Blocks; launch 별도 |

`svd_steps`의 energy_fraction은 입력 제곱합 보존 비율이며 추천 정확도가 아닙니다. 영행렬에서는 None입니다. `symmetric_eigen`을 일반 비대칭행렬에 쓰지 마세요.


In [ ]:
import os, sys, subprocess
os.environ['GRADIO_ANALYTICS_ENABLED']='False'
IN_COLAB='google.colab' in sys.modules
COURSE_VERSION='2026-fall-w06b'
if IN_COLAB:
    subprocess.run([sys.executable,'-m','pip','install','-q',
        f'luna-recommender-course[apps] @ git+https://github.com/lunalab-ai/recommender.git@{COURSE_VERSION}',
        'gradio==6.26.0'],check=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from luna_recsys.svd_foundations import teaching_matrix, unit_vector, symmetric_eigen, svd_steps, transform_stages
from luna_recsys.svd_lab import direction_view, stages_view, reconstruction_view, build_svd_app
from luna_recsys.model_comparison import definition_link
np.set_printoptions(precision=6,suppress=True)
print('Python',sys.version.split()[0], '고정 수업 버전',COURSE_VERSION)


In [ ]:
apis=[teaching_matrix,unit_vector,symmetric_eigen,svd_steps,transform_stages,
      direction_view,stages_view,reconstruction_view,build_svd_app,definition_link]
display(Markdown('\n'.join(f'- [{f.__qualname__}]({definition_link(f,COURSE_VERSION)})' for f in apis)))


## 1. 행렬곱을 숫자로 읽기

`A @ x`의 첫 좌표는 첫 행과 입력 좌표의 곱을 더한 값입니다. `A * x`는 원소별 곱으로 다른 모양의 결과를 만듭니다. 먼저 종이에 계산하고 실행하세요.


### A1 · 곱셈 예측

목표: 앞의 계산을 자신의 식과 문장으로 재현합니다. 힌트: 해당 절의 중간 행렬·차원·검산식을 확인하세요. 자가 점검: 수치 답은 원래 식에 대입하고, 서술 답은 코드의 실제 수치 한 가지를 근거로 드세요.


In [ ]:
a1_prediction=None
a1_reason=''


In [ ]:
small=np.array([[2.,1.],[0.,1.]])
x=np.array([1.,2.])
print('행렬곱:',small@x,'shape:',(small@x).shape)
print('원소별 곱:\n',small*x,'shape:',(small*x).shape)
print('전치:\n',small.T,'Gram:\n',small.T@small)
assert np.allclose(small@x,[4,2])


## 2. 길이1과 특별한 방향

$v=(1,1)^\top$의 길이는 $\sqrt{2}$입니다. 두 좌표를 모두 $\sqrt{2}$로 나누면 방향은 같고 길이만1이 됩니다. $Bv=\lambda v$가 성립하는지 양쪽을 직접 비교합니다.


In [ ]:
B=np.array([[2.,1.],[1.,2.]])
v1=unit_vector([1,1]);v2=unit_vector([1,-1])
print('단위벡터:',v1,v2,'내적:',v1@v2)
print('Bv1:',B@v1,'3v1:',3*v1)
print('Bv2:',B@v2,'1v2:',v2)
assert np.allclose(B@v1,3*v1)
assert np.allclose(B@v2,v2)
eig=symmetric_eigen(B)
values,vectors,residual=eig['values'],eig['vectors'],eig['residual']
print('내림차순 고유값:',values,'열별 고유벡터:\n',vectors)
print('대입 검산 오차:',residual)
t,table,fig=direction_view('main',45);print(t);display(table);display(fig)


### A2 · 고유벡터 정규화 디버깅: (1,1)을 2로 나누면 단위벡터인가? 올바른 분모와 이유를 쓰세요.

목표: 앞의 계산을 자신의 식과 문장으로 재현합니다. 힌트: 해당 절의 중간 행렬·차원·검산식을 확인하세요. 자가 점검: 수치 답은 원래 식에 대입하고, 서술 답은 코드의 실제 수치 한 가지를 근거로 드세요.


In [ ]:
a2_denominator=None
a2_reason=''


## 3. 대표 행렬의 SVD를 한 단계씩

$A=\begin{bmatrix}5&1\\1&5\end{bmatrix}$에 대해 아래 셀을 실행하기 전 $A^\top A$의 네 원소를 구하세요. 고유값36,16은 **Gram 행렬의 고유값**이고 특이값은 그 제곱근6,4입니다. 고유벡터는 배열의 열입니다.


In [ ]:
A=teaching_matrix('main')
G=A.T@A
eig=symmetric_eigen(G)
lam,V=eig['values'],eig['vectors']
sigma=np.sqrt(lam)
U_by_hand=(A@V)/sigma[np.newaxis,:]
Sigma=np.diag(sigma)
for name,value in [('A',A),('A.T @ A',G),('lambda',lam),('V',V),('sigma',sigma),('U=Av/sigma',U_by_hand),('Sigma',Sigma),('V.T',V.T)]:
    print(name,'shape=',value.shape,'\n',value)
rebuilt=U_by_hand@Sigma@V.T
print('최종 재구성:',rebuilt)
assert np.allclose(G,[[26,10],[10,26]])
assert np.allclose(lam,[36,16])
assert np.allclose(sigma,[6,4])
assert np.allclose(U_by_hand.T@U_by_hand,np.eye(2))
assert np.allclose(rebuilt,A)


### A3 · 고유값→특이값 빈칸: [36,16]에서 두 특이값을 구하고 제곱근을 쓰는 이유를 설명하세요.

목표: 앞의 계산을 자신의 식과 문장으로 재현합니다. 힌트: 해당 절의 중간 행렬·차원·검산식을 확인하세요. 자가 점검: 수치 답은 원래 식에 대입하고, 서술 답은 코드의 실제 수치 한 가지를 근거로 드세요.


In [ ]:
a3_sigma=None
a3_reason=''


## 4. 라이브러리 반환값과 검산

`np.linalg.svd(A, full_matrices=False)`는 U, **1차원 특이값 배열 s**, **V의 전치 Vh**를 반환합니다. 다시 전치하면 틀린 곱이 됩니다. 부호가 다른 답도 양쪽 인수의 부호가 함께 바뀌었다면 옳을 수 있습니다.


In [ ]:
U,s,Vh=np.linalg.svd(A,full_matrices=False)
print('U/s/Vh shapes:',U.shape,s.shape,Vh.shape)
print('재구성:\n',U@np.diag(s)@Vh)
assert np.allclose(U@np.diag(s)@Vh,A)
assert np.allclose(A@Vh.T,U*s)
assert np.allclose(G@Vh.T,Vh.T*(s*s))
# 첫 축의 양쪽 부호를 동시에 바꿔도 동일한 A
U_alt=U.copy();Vh_alt=Vh.copy();U_alt[:,0]*=-1;Vh_alt[0,:]*=-1
assert np.allclose(U_alt@np.diag(s)@Vh_alt,A)
print('서로 다른 부호도 재구성과 직교성이 맞으면 올바릅니다.')
text,fig=stages_view();print(text);display(fig)


### A4 · 코드 오류 찾기: U * s * Vh를 왜 재구성이라고 부르면 안 되는가? 올바른 식을 쓰세요.

목표: 앞의 계산을 자신의 식과 문장으로 재현합니다. 힌트: 해당 절의 중간 행렬·차원·검산식을 확인하세요. 자가 점검: 수치 답은 원래 식에 대입하고, 서술 답은 코드의 실제 수치 한 가지를 근거로 드세요.


In [ ]:
a4_reconstruction=None
a4_reason=''


### A5 · 부호가 다른 답: 첫 U열만 -1배하고 Vh는 그대로 두어도 되는가?

목표: 앞의 계산을 자신의 식과 문장으로 재현합니다. 힌트: 해당 절의 중간 행렬·차원·검산식을 확인하세요. 자가 점검: 수치 답은 원래 식에 대입하고, 서술 답은 코드의 실제 수치 한 가지를 근거로 드세요.


In [ ]:
a5_reason=''


## 5. 오늘의 확인과 다음 실습

1. 고유벡터가 0벡터이면 왜 방향 설명이 무의미한가?
2. 고유값이 -2이면 길이와 방향은 어떻게 변하는가?
3. SVD는 입력 벡터에 어느 인수부터 작용하는가?
4. 특이값은 A의 고유값인가, AᵀA의 고유값의 제곱근인가?
5. U와 V의 부호가 책과 다를 때 무엇을 검산해야 하는가?

계산 문제편 Q1–Q5를 풀고 상세 풀이와 비교하세요. 실습 B에서는 큰 패턴만 남기는 의미와 추천 오차의 차이를 확인합니다.


In [ ]:
check=svd_steps(teaching_matrix(),1)
assert np.allclose(check['s'],[6,4])
assert np.allclose(check['reconstruction'],3)
assert np.isclose(check['error'],4)
assert np.allclose(U_by_hand@Sigma@V.T,A)
assert symmetric_eigen(B)['residual']<1e-10
print('W06B-A RUNTIME CHECK PASSED: Gram, eigen, SVD, reconstruction.')
